# Task 1 - Data Understanding and Preparation

This notebook follows the KDD process for the first project: data loading, merge validation, quality assessment, exploratory analysis, and feature engineering.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

import functions as uf

%load_ext autoreload
%autoreload 2

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
sns.set_theme(style="whitegrid", context="talk")

candidate_dirs = [Path.cwd().parent / "dataset", Path.cwd() / "dataset"]
DATA_DIR = next((path for path in candidate_dirs if (path / "artists.csv").exists() and (path / "tracks.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not locate the repo-level dataset/ directory.")

print(f"Using data directory: {DATA_DIR.resolve()}")


## Load The Raw Datasets

`artists.csv` is semicolon-separated, while `tracks.csv` is comma-separated and includes quoted multiline lyrics.


In [ ]:
df_artist = pd.read_csv(DATA_DIR / "artists.csv", sep=";")
df_track = pd.read_csv(DATA_DIR / "tracks.csv", sep=",")

print(f"Artists shape: {df_artist.shape}")
print(f"Tracks shape: {df_track.shape}")

display(df_artist.head())
display(df_track.head(2))


## Merge Validation And Dataset Overview

We merge artist-level and song-level data using the artist identifier, then verify coverage and summarize the variables by semantic role.


In [ ]:
df = pd.merge(
    df_artist,
    df_track,
    left_on="id_author",
    right_on="id_artist",
    validate="one_to_many",
).rename(columns={"id": "id_song"})

merge_overview = pd.DataFrame(
    {
        "metric": [
            "artist_rows",
            "track_rows",
            "merged_rows",
            "unique_artists_in_artists",
            "unique_artists_in_tracks",
            "unique_artists_after_merge",
            "unique_song_ids_after_merge",
            "artist_features",
            "track_features",
            "merged_features",
            "merge_coverage_pct",
        ],
        "value": [
            len(df_artist),
            len(df_track),
            len(df),
            df_artist["id_author"].nunique(),
            df_track["id_artist"].nunique(),
            df["id_author"].nunique(),
            df["id_song"].nunique(),
            df_artist.shape[1],
            df_track.shape[1],
            df.shape[1],
            round(100 * len(df) / len(df_track), 2),
        ],
    }
)

display(merge_overview)
print(f"Expected working dataset: {len(df)} rows over {df['id_author'].nunique()} artists.")


In [ ]:
feature_group_table = uf.feature_group_table(df).sort_values(["level", "group", "feature"]).reset_index(drop=True)
feature_group_summary = (
    feature_group_table.groupby(["level", "group"])
    .size()
    .rename("n_features")
    .reset_index()
)
engineered_docs = uf.engineered_feature_table()

display(feature_group_summary)
display(feature_group_table)
display(engineered_docs)


## Redundant Text Fields

Before deeper analysis, we check the overlap between duplicated textual columns and keep the richer representation when redundancy is high.


In [ ]:
print(f"NaN in 'album': {df['album'].isna().sum()}")
print(f"NaN in 'album_name': {df['album_name'].isna().sum()}")
album_comparison, album_score = uf.compare_text_columns(df, "album", "album_name")

title_comparison, title_score = uf.compare_text_columns(df, "title", "full_title")

redundancy_summary = pd.DataFrame(
    [
        {
            "pair": "album vs album_name",
            "rows_evaluated": len(album_comparison),
            "avg_similarity": round(album_score, 2),
            "decision": "drop album",
            "motivation": "album_name has far fewer missing values",
        },
        {
            "pair": "title vs full_title",
            "rows_evaluated": len(title_comparison),
            "avg_similarity": round(title_score, 2),
            "decision": "drop title",
            "motivation": "full_title preserves featured-artist context",
        },
    ]
)

display(redundancy_summary)
display(album_comparison.nsmallest(10, "similarity_score"))
display(title_comparison.nsmallest(10, "similarity_score"))

df = df.drop(columns=["album", "title"])


## Cleaning, Typing, And Derived Dates

We standardize missing values, coerce numeric/date columns, validate boolean fields, and derive a cleaned release year for later analysis.


In [ ]:
df = df.replace(r"^\s*$", np.nan, regex=True)

numeric_columns = [
    "stats_pageviews",
    "swear_IT",
    "swear_EN",
    "year",
    "month",
    "day",
    "n_sentences",
    "n_tokens",
    "tokens_per_sent",
    "char_per_tok",
    "lexical_density",
    "avg_token_per_clause",
    "bpm",
    "centroid",
    "rolloff",
    "flux",
    "rms",
    "zcr",
    "flatness",
    "spectral_complexity",
    "pitch",
    "loudness",
    "disc_number",
    "track_number",
    "duration_ms",
    "popularity",
    "modified_popularity",
    "latitude",
    "longitude",
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

for column in ["birth_date", "active_start", "active_end", "album_release_date"]:
    df[column] = pd.to_datetime(df[column], errors="coerce")

explicit_map = {True: True, False: False, "True": True, "False": False}
df["explicit"] = df["explicit"].map(explicit_map)
df["explicit_numeric"] = df["explicit"].astype("float")

current_year = pd.Timestamp.today().year
plausible_track_year = df["year"].between(1900, current_year + 1)
plausible_album_year = df["album_release_date"].dt.year.between(1900, current_year + 1)

df["track_release_date"] = pd.to_datetime(
    {
        "year": df["year"],
        "month": df["month"],
        "day": df["day"],
    },
    errors="coerce",
)

df["release_year"] = df["year"].where(plausible_track_year)
df["release_year"] = df["release_year"].fillna(df["album_release_date"].dt.year.where(plausible_album_year))

display(df[["birth_date", "active_start", "active_end", "album_release_date", "track_release_date", "release_year"]].head())


## Missing Values

The table below summarizes missingness by feature and by semantic group so that later modeling choices can separate structural missingness from optional metadata.


In [ ]:
feature_lookup = uf.feature_group_table(df).drop_duplicates(subset=["feature"]).set_index("feature")
missing_summary = pd.DataFrame(
    {
        "feature": df.columns,
        "missing_count": df.isna().sum().values,
    }
)
missing_summary["missing_pct"] = (100 * missing_summary["missing_count"] / len(df)).round(2)
missing_summary = missing_summary.join(feature_lookup[["group", "level"]], on="feature")
missing_summary = missing_summary.fillna({"group": "derived", "level": "song"})
missing_summary = missing_summary.sort_values(["missing_pct", "missing_count"], ascending=False).reset_index(drop=True)

group_missing = (
    missing_summary.groupby(["level", "group"])[["missing_count", "missing_pct"]]
    .mean()
    .rename(columns={"missing_count": "avg_missing_count", "missing_pct": "avg_missing_pct"})
    .reset_index()
    .sort_values("avg_missing_pct", ascending=False)
)

display(missing_summary)
display(group_missing)

fig, ax = plt.subplots(figsize=(10, 6))
missing_summary.head(15).sort_values("missing_pct").plot.barh(
    x="feature",
    y="missing_pct",
    ax=ax,
    legend=False,
    color="#c44e52",
)
ax.set_title("Top 15 Missing Features (%)")
ax.set_xlabel("Missing percentage")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## Duplicates And Inconsistent Values

We inspect duplicate identifiers, repeated merged rows, repeated artist-title combinations, and potentially inconsistent categorical labels.


In [ ]:
duplicate_song_id_mask = df["id_song"].duplicated(keep=False)
duplicate_row_mask = df.duplicated(keep=False)
duplicate_artist_title_mask = df.duplicated(subset=["name", "full_title"], keep=False)

normalized_name_artist = df["name_artist"].map(uf.normalize_text)
normalized_name = df["name"].map(uf.normalize_text)
artist_name_mismatch_mask = normalized_name_artist != normalized_name

duplicates_summary = pd.DataFrame(
    [
        {"check": "duplicated_song_ids", "count": int(duplicate_song_id_mask.sum())},
        {"check": "fully_duplicated_rows", "count": int(duplicate_row_mask.sum())},
        {"check": "duplicated_artist_full_title_pairs", "count": int(duplicate_artist_title_mask.sum())},
        {"check": "artist_name_mismatches", "count": int(artist_name_mismatch_mask.sum())},
    ]
)

display(duplicates_summary)
display(
    df.loc[duplicate_song_id_mask, ["id_song", "name", "full_title", "album_name", "release_year"]]
    .sort_values(["id_song", "release_year"])
    .head(20)
)
display(
    df.loc[artist_name_mismatch_mask, ["name", "name_artist", "full_title"]]
    .drop_duplicates()
    .head(20)
)


In [ ]:
for column in ["language", "album_type", "explicit", "region", "country", "nationality"]:
    print(f"\nValue counts for {column}:")
    display(df[column].value_counts(dropna=False).rename("count").to_frame().head(15))


## Incorrect Dates And Extreme Values (OOD)

We flag impossible dates, implausible years, and suspicious numeric values using both domain rules and robust IQR-based summaries.


In [ ]:
current_year = pd.Timestamp.today().year
partial_track_date_mask = df[["year", "month", "day"]].notna().any(axis=1)

invalid_date_summary = pd.DataFrame(
    [
        {"check": "birth_date_in_future", "count": int((df["birth_date"] > pd.Timestamp.today()).sum())},
        {"check": "active_start_in_future", "count": int((df["active_start"] > pd.Timestamp.today()).sum())},
        {"check": "active_end_before_active_start", "count": int((df["active_end"] < df["active_start"]).sum())},
        {"check": "album_release_date_in_future", "count": int((df["album_release_date"] > pd.Timestamp.today()).sum())},
        {"check": "track_year_outside_1900_to_current+1", "count": int((df["year"].notna() & ~df["year"].between(1900, current_year + 1)).sum())},
        {"check": "invalid_track_release_date_from_parts", "count": int((partial_track_date_mask & df["track_release_date"].isna()).sum())},
        {"check": "release_year_missing_after_cleaning", "count": int(df["release_year"].isna().sum())},
    ]
)

domain_rules = {
    "popularity": lambda s: (s < 0) | (s > 100),
    "modified_popularity": lambda s: s < 0,
    "stats_pageviews": lambda s: s < 0,
    "bpm": lambda s: (s <= 0) | (s > 300),
    "duration_ms": lambda s: (s <= 0) | (s > 900000),
    "tokens_per_sent": lambda s: s <= 0,
    "avg_token_per_clause": lambda s: s < 0,
    "lexical_density": lambda s: (s < 0) | (s > 1),
    "year": lambda s: (s < 1900) | (s > current_year + 1),
}

outlier_summary = uf.summarize_outliers(
    df,
    columns=[
        "popularity",
        "modified_popularity",
        "stats_pageviews",
        "bpm",
        "duration_ms",
        "tokens_per_sent",
        "avg_token_per_clause",
        "lexical_density",
        "year",
    ],
    domain_rules=domain_rules,
)

suspicious_counts = pd.DataFrame(
    [
        {"check": "year_equals_2100", "count": int((df["year"] == 2100).sum())},
        {"check": "negative_popularity", "count": int((df["popularity"] < 0).sum())},
        {"check": "popularity_above_100", "count": int((df["popularity"] > 100).sum())},
        {"check": "bpm_above_300", "count": int((df["bpm"] > 300).sum())},
        {"check": "duration_above_15_minutes", "count": int((df["duration_ms"] > 900000).sum())},
    ]
)

display(invalid_date_summary)
display(outlier_summary)
display(suspicious_counts)


## Feature Engineering

We add three new features aligned with the project goals: profanity density, macro-area regionality, and a sentence complexity index.


In [ ]:
df["swear_total"] = df[["swear_IT", "swear_EN"]].fillna(0).sum(axis=1)
df["swear_density_total"] = uf.safe_divide(df["swear_total"], df["n_tokens"])
df["artist_macroarea"] = df["region"].apply(uf.map_region_to_macroarea)
df["sentence_complexity_index"] = uf.build_weighted_zscore(
    df,
    columns=["tokens_per_sent", "avg_token_per_clause", "lexical_density", "char_per_tok"],
    weights=[1, 1, 1, 1],
    min_non_missing=3,
)

engineered_additions = pd.DataFrame(
    [
        {
            "feature": "swear_density_total",
            "formula": "(swear_IT + swear_EN) / n_tokens",
            "motivation": "Compare profanity across songs with different lyric lengths.",
        },
        {
            "feature": "artist_macroarea",
            "formula": "region -> {North, Center, South, Islands, Missing}",
            "motivation": "Stabilize the regional comparison by reducing sparsity.",
        },
        {
            "feature": "sentence_complexity_index",
            "formula": "mean z-score of tokens_per_sent, avg_token_per_clause, lexical_density, char_per_tok",
            "motivation": "Capture a broader notion of textual complexity than a single proxy.",
        },
    ]
)

display(engineered_additions)
display(df[["swear_total", "swear_density_total", "artist_macroarea", "sentence_complexity_index"]].describe(include="all").T)
display(df["artist_macroarea"].value_counts(dropna=False).rename("song_count").to_frame())


## Distribution Analysis

We inspect the main project variables with histograms, boxplots, and violin plots. For heavily skewed variables we also inspect a `log1p` transformed view.


In [ ]:
def plot_distribution_triplet(data, column, pretty_name=None, log1p=False):
    series = data[column].dropna()
    if series.empty:
        print(f"Skipping {column}: no valid values.")
        return

    label = pretty_name or column
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    sns.histplot(series, bins=40, kde=True, ax=axes[0], color="#4c72b0")
    axes[0].set_title(f"Histogram - {label}")

    sns.boxplot(x=series, ax=axes[1], color="#dd8452")
    axes[1].set_title(f"Boxplot - {label}")

    sns.violinplot(x=series, ax=axes[2], color="#55a868", inner="quartile")
    axes[2].set_title(f"Violin plot - {label}")

    plt.tight_layout()
    plt.show()

    if log1p:
        non_negative = series[series >= 0]
        if not non_negative.empty:
            fig, ax = plt.subplots(figsize=(8, 4))
            sns.histplot(np.log1p(non_negative), bins=40, kde=True, ax=ax, color="#8172b3")
            ax.set_title(f"Log1p histogram - {label}")
            ax.set_xlabel(f"log1p({label})")
            plt.tight_layout()
            plt.show()

variables_to_plot = [
    ("popularity", "Popularity", True),
    ("stats_pageviews", "Pageviews", True),
    ("bpm", "BPM", False),
    ("duration_ms", "Duration (ms)", True),
    ("lexical_density", "Lexical density", False),
    ("swear_IT", "Italian swear count", False),
    ("swear_EN", "English swear count", False),
    ("swear_total", "Total swear count", False),
    ("release_year", "Release year", False),
]

for column, label, use_log in variables_to_plot:
    plot_distribution_triplet(df, column, pretty_name=label, log1p=use_log)


## Swear Density And Regionality

This section connects the custom feature engineering to the project question by comparing profanity density across artist macro-areas.


In [ ]:
macroarea_order = ["North", "Center", "South", "Islands", "Missing"]
macroarea_stats = (
    df.groupby("artist_macroarea")["swear_density_total"]
    .agg(["count", "mean", "median", "std"])
    .reindex(macroarea_order)
)

display(macroarea_stats)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
sns.boxplot(
    data=df,
    x="artist_macroarea",
    y="swear_density_total",
    order=macroarea_order,
    ax=axes[0],
    color="#4c72b0",
)
axes[0].set_title("Swear density by macro-area")
axes[0].set_xlabel("Artist macro-area")
axes[0].set_ylabel("Swear density")
axes[0].tick_params(axis="x", rotation=20)

sns.violinplot(
    data=df,
    x="artist_macroarea",
    y="swear_density_total",
    order=macroarea_order,
    ax=axes[1],
    color="#55a868",
    inner="quartile",
    cut=0,
)
axes[1].set_title("Distribution of swear density by macro-area")
axes[1].set_xlabel("Artist macro-area")
axes[1].set_ylabel("Swear density")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

coverage_by_macroarea = df["artist_macroarea"].value_counts().reindex(macroarea_order)
display(coverage_by_macroarea.rename("song_count").to_frame())
print("Interpretation note: any difference across macro-areas should be discussed together with artist coverage imbalance and the Missing group.")


## Correlation And Redundancy Analysis

We restrict the correlation matrices to cleaned numeric variables, compare Pearson and Spearman correlation, and highlight strongly related pairs.


In [ ]:
correlation_columns = [
    "popularity",
    "modified_popularity",
    "stats_pageviews",
    "explicit_numeric",
    "bpm",
    "duration_ms",
    "release_year",
    "swear_IT",
    "swear_EN",
    "swear_total",
    "swear_density_total",
    "n_sentences",
    "n_tokens",
    "tokens_per_sent",
    "char_per_tok",
    "lexical_density",
    "avg_token_per_clause",
    "sentence_complexity_index",
    "centroid",
    "rolloff",
    "flux",
    "rms",
    "zcr",
    "flatness",
    "spectral_complexity",
    "pitch",
    "loudness",
]

corr_df = df[correlation_columns].copy()
pearson_corr = corr_df.corr(method="pearson")
spearman_corr = corr_df.corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(24, 10))
sns.heatmap(pearson_corr, cmap="coolwarm", center=0, ax=axes[0])
axes[0].set_title("Pearson correlation matrix")

sns.heatmap(spearman_corr, cmap="coolwarm", center=0, ax=axes[1])
axes[1].set_title("Spearman correlation matrix")

plt.tight_layout()
plt.show()

def top_correlation_pairs(corr_matrix, threshold=0.7):
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    pairs = (
        upper.stack()
        .rename("correlation")
        .reset_index()
        .rename(columns={"level_0": "feature_a", "level_1": "feature_b"})
    )
    return pairs.loc[pairs["correlation"].abs() >= threshold].sort_values(
        "correlation",
        key=lambda s: s.abs(),
        ascending=False,
    )

high_corr_pairs = top_correlation_pairs(pearson_corr, threshold=0.7)
focus_features = [
    "popularity",
    "stats_pageviews",
    "explicit_numeric",
    "swear_density_total",
    "sentence_complexity_index",
    "bpm",
    "duration_ms",
    "release_year",
]

focus_corr = pearson_corr.loc[focus_features, corr_df.columns].T.sort_index()

display(high_corr_pairs)
display(focus_corr)


## Discussion Notes For The Report

When writing the final report, discuss both descriptive findings and data limitations:

- `album` / `album_name` and `title` / `full_title` are redundancy candidates, but the richer text version was retained.
- Geography is informative but incomplete, so regional findings must be framed cautiously.
- Popularity and pageviews are likely skewed and should be interpreted with outlier awareness.
- Swear density is more comparable than raw swear counts because it adjusts for lyric length.
- The sentence complexity index is intentionally transparent: higher values mean longer, denser, and lexically richer sentences.
